<span style="color:red">

# Desafio — CD&IA
## Registro Hospitalar de Câncer de São Paulo (RHC/SP)

</span>



## Importando as bibliotecas necessárias

- `pandas` para análise dos dados, criação de DataFrames e manipulação de colunas e linhas.
- `numpy` para operações matemáticas.
- `dbfread` para leitura da fonte de dados, necessária pois o arquivo está no formato `.dbf`,
  formato legado amplamente utilizado em bases de dados governamentais brasileiras.

In [1]:
import pandas as pd
import numpy as np
from dbfread import DBF
from sklearn.preprocessing import StandardScaler

## Carregamento dos dados

- `encoding="latin1"` para garantir a leitura correta de caracteres especiais e acentos presentes em dados governamentais brasileiros.
- `list()` para converter o objeto `.dbf` em uma lista de registros legível pelo pandas.
- Salvamento em `.parquet` com `index=False` para otimizar carregamentos futuros sem salvar o índice do DataFrame.
- Recarregamento a partir do `.parquet` para garantir a integridade dos tipos de dados.

In [2]:
dados = DBF("RHC_2000_2025_GERAL.DBF", encoding="latin1")
df = pd.DataFrame(list(dados))

df.to_parquet("rhc.parquet", index=False)


In [3]:
df = pd.read_parquet("rhc.parquet")

display(df.head())
print(f"Dataset carregado: {df.shape[0]} linhas e {df.shape[1]} colunas")

,INSTITU,ESCOLARI,IDADE,SEXO,UFNASC,UFRESID,IBGE,CIDADE,CATEATEND,DTCONSULT,...,HISTORICOC,RECESTROG,RECEPROG,RECHER2,P16IHQ,HABILIT,HABIT11,HABILIT1,HABILIT2,CIDADEH
0,016675,2,100,2,SP,SP,3550308,SAO PAULO,9,2000-06-09,...,NaN,NaN,NaN,NaN,NaN,7,CACON com Serviço de Oncologia Pediátrica,3,2,São Paulo
1,020737,1,100,1,MG,MG,3133402,ITAPAGIPE,9,2000-01-03,...,NaN,NaN,NaN,NaN,NaN,7,CACON com Serviço de Oncologia Pediátrica,3,2,Barretos
2,020737,1,100,1,MG,MG,3133402,ITAPAGIPE,9,2000-01-03,...,NaN,NaN,NaN,NaN,NaN,7,CACON com Serviço de Oncologia Pediátrica,3,2,Barretos
3,020737,1,100,1,MG,MG,3133402,ITAPAGIPE,9,2002-04-29,...,NaN,NaN,NaN,NaN,NaN,7,CACON com Serviço de Oncologia Pediátrica,3,2,Barretos
4,020737,1,105,2,SP,SP,3546603,SANTA FE DO SUL,9,2005-09-13,...,NaN,NaN,NaN,NaN,NaN,7,CACON com Serviço de Oncologia Pediátrica,3,2,Barretos


Dataset carregado: 1344819 linhas e 114 colunas


## Preparação dos dados

A cada etapa, `.shape[0]` é utilizado para validar o resultado do filtro aplicado,
exibindo o número de pacientes restantes após cada seleção.

**Filtro 1 — Topografia de pulmão (CID-10: C34)**
Seleciona apenas tumores malignos dos brônquios e pulmões.

In [4]:
# 1. Selecionar pacientes com Topografia de pulmão (TOPOGRUP = C34);
df = df[df["TOPOGRUP"] == "C34"]
print(f"1. Após filtro C34: {df.shape[0]} pacientes")

1. Após filtro C34: 62471 pacientes


**Filtro 2 — Residência em São Paulo**
Restringe a análise a pacientes residentes no estado de SP.

In [5]:
# 2. Selecionar pacientes com estado de Residência de São Paulo (UFRESID = SP);
df = df[df["UFRESID"] == "SP"]
print(f"2. Após filtro SP: {df.shape[0]} pacientes")

2. Após filtro SP: 57346 pacientes


**Filtro 3 — Confirmação microscópica (BASEDIAG = 3)**
Garante que apenas diagnósticos com confirmação laboratorial sejam incluídos,
aumentando a confiabilidade dos dados.

In [6]:
# 3. Selecionar pacientes com Base do Diagnóstico com Confirmação Microscópica (BASEDIAG = 3);
df = df[df["BASEDIAG"] == 3]
print(f"3. Após filtro BASEDIAG: {df.shape[0]} pacientes")

3. Após filtro BASEDIAG: 55551 pacientes


**Filtro 4 — `.isin()` e `~`**
- `.isin([...])` retorna `True` para cada linha onde o valor está na lista
- `~` inverte o resultado — mantém apenas quem **não** está na lista
- `0` e `"0"` cobrem os dois casos: coluna numérica ou texto

In [7]:
# Retirar categorias 0, X e Y da coluna ECGRUP;
df = df[~df["ECGRUP"].isin([0, "0", "X", "Y"])]
print(f"4. Após filtro da retirada de 0, X e Y: {df.shape[0]} pacientes")

4. Após filtro da retirada de 0, X e Y: 49633 pacientes


**Filtro 5 — condições combinadas com `&`**
- `&` exige que as duas condições sejam verdadeiras simultaneamente
- `~` inverte — retira apenas quem fez **os dois** tratamentos juntos
- Quem fez só um dos dois permanece no dataset

In [8]:
# Retirar pacientes que fizeram Hormonioterapia e TMO (HORMONIO = 1 e TMO = 1);
df = df[~((df["HORMONIO"] == 1) & (df["TMO"] == 1))]
print(f"5. Após filtro da retirada de Hormonioterapia E TMO: {df.shape[0]} pacientes")

5. Após filtro da retirada de Hormonioterapia E TMO: 49633 pacientes


In [9]:
# Selecionar pacientes com Ano de Diagnóstico até 2019 (ANODIAG <= 2019);
df = df[df["ANODIAG"] <= 2019]
print(f"6. Após filtro até 2019: {df.shape[0]} pacientes")

6. Após filtro até 2019: 38998 pacientes


In [10]:
# 7. Retirar pacientes com IDADE menor do que 20 anos;
df = df[df["IDADE"] >= 20]
print(f"7. Após filtro menor que 20 anos: {df.shape[0]} pacientes")

7. Após filtro menor que 20 anos: 38985 pacientes


## Etapa 8 — Cálculo e codificação de intervalos de tempo

A partir das datas de consulta, diagnóstico e tratamento, são calculadas três diferenças em dias
e codificadas em categorias numéricas para uso no modelo.

**Colunas calculadas:**
- `CONSDIAG` — dias entre consulta e diagnóstico → 0: até 30 | 1: 31–60 | 2: +60 | 3: negativo
- `DIAGTRAT` — dias entre diagnóstico e tratamento → 0: até 60 | 1: 61–90 | 2: +90 | 3: não tratou
- `TRATCONS` — dias entre tratamento e consulta → 0: até 60 | 1: 61–90 | 2: +90 | 3: não tratou

`errors="coerce"` converte datas inválidas em `NaT` e `np.select` aplica as condições em cascata,
atribuindo a categoria da primeira condição verdadeira.

In [11]:
# 8. Calcular a diferença em dias entre Diagnóstico e Consulta, Tratamento e Diagnóstico,
# Tratamento e Consulta, a partir dessas 3 datas (DTCONSULT, DTDIAG e DTTRAT).
# Após o cálculo, codificar as colunas da seguinte forma:

df["DTCONSULT"] = pd.to_datetime(df["DTCONSULT"], errors="coerce")
df["DTDIAG"]    = pd.to_datetime(df["DTDIAG"],    errors="coerce")
df["DTTRAT"]    = pd.to_datetime(df["DTTRAT"],    errors="coerce")

df["diff_CONSDIAG"] = (df["DTCONSULT"] - df["DTDIAG"]).dt.days
df["diff_DIAGTRAT"] = (df["DTTRAT"]    - df["DTDIAG"]).dt.days
df["diff_TRATCONS"] = (df["DTTRAT"]    - df["DTCONSULT"]).dt.days

# CONSDIAG - 0 = até 30 dias; 1 = entre 31 e 60 dias; 2 = mais de 60 dias; 3 para diferenças negativas;
conditions = [
    df["diff_CONSDIAG"] < 0,
    df["diff_CONSDIAG"] <= 30,
    df["diff_CONSDIAG"] <= 60,
    df["diff_CONSDIAG"] > 60
]
choices = [3, 0, 1, 2]
df["CONSDIAG"] = np.select(conditions, choices, default=np.nan)

#  DIAGTRAT - 0 = até 60 dias; 1 = entre 61 e 90 dias; 2 = mais de 90 dias; 3 = Não tratou (datas de tratamento vazias);
conditions2 = [
    df["diff_DIAGTRAT"].isna(),
    df["diff_DIAGTRAT"] <= 60,
    df["diff_DIAGTRAT"] <= 90,
    df["diff_DIAGTRAT"] > 90
]
df["DIAGTRAT"] = np.select(conditions2, choices, np.nan)

# TRATCONS - 0 = até 60 dias; 1 = entre 61 e 90 dias; 2 = mais de 90 dias; 3 = Não tratou (datas de tratamento vazias).
conditions3 = [
    df["diff_TRATCONS"].isna(),
    df["diff_TRATCONS"] <= 60,
    df["diff_TRATCONS"] <= 90,
    df["diff_TRATCONS"] > 90
]
df["TRATCONS"] = np.select(conditions3, choices, default=np.nan)

# remove colunas temporárias
df = df.drop(columns=["diff_CONSDIAG", "diff_DIAGTRAT", "diff_TRATCONS"])

print("CONSDIAG:\n", df["CONSDIAG"].value_counts())
print("\nDIAGTRAT:\n", df["DIAGTRAT"].value_counts())
print("\nTRATCONS:\n", df["TRATCONS"].value_counts())

CONSDIAG:
 CONSDIAG
3.0    22447
0.0    11437
1.0     3166
2.0     1935
Name: count, dtype: int64

DIAGTRAT:
 DIAGTRAT
0.0    23748
3.0     6419
2.0     4771
1.0     4047
Name: count, dtype: int64

TRATCONS:
 TRATCONS
0.0    22772
3.0     6419
2.0     5640
1.0     4154
Name: count, dtype: int64


**Etapa 9 — Extração de números das colunas DRS e DRS_INST**

As colunas `DRS` e `DRS_INST` contém texto junto com o número identificador.
`str.extract()` com regex `(\d+)` isola apenas os dígitos, descartando o restante.

In [12]:
# 9. Extrair somente o número das colunas DRS e DRS INST;

df["DRS"] = df["DRS"].str.extract(r"(\d+)")
df["DRS_INST"] = df["DRS_INST"].str.extract(r"(\d+)")

print("DRS:\n", df["DRS"].value_counts())
print("\nDRS_INST:\n", df["DRS_INST"].value_counts())

DRS:
 DRS
01    18205
15     2736
07     2698
06     2564
13     1734
10     1493
17     1330
09     1221
03     1197
16     1047
05     1030
02      858
08      724
04      720
14      717
11      539
12       76
Name: count, dtype: int64

DRS_INST:
 DRS_INST
01    19417
06     4418
05     4036
07     2729
15     1826
13     1559
10     1198
17     1143
03      558
09      511
04      408
11      342
02      300
16      242
08      216
14       69
12       13
Name: count, dtype: int64


**Etapa 10 — Criação da coluna de óbito**

`.isin([3, 4])` identifica as categorias que representam óbito e `.astype(int)` converte
o resultado `True/False` para binário `1/0`.

In [13]:
# 10. Criar a coluna binária de óbito, a partir da coluna ULTINFO, onde as categorias 1 e
# 2 representam que o paciente está vivo e as 3 e 4 representam o óbito por qualquer
# motivo;

df["obito"] = df["ULTINFO"].isin([3, 4]).astype(int)
print(df["obito"].value_counts())

obito
1    36531
0     2454
Name: count, dtype: int64


**Etapa 11 — Remoção de colunas**

Colunas irrelevantes para o modelo são removidas com `.drop()`.  
`errors="ignore"` evita erros caso alguma coluna da lista não exista no DataFrame.

In [14]:
# 11. Retirar as colunas:
# UFNASC, UFRESID, CIDADE, DTCONSULT, CLINICA , DTDIAG,
# BASEDIAG, TOPOGRUP, DESCTOPO, ...

colunas_remover = [
    "UFNASC", "UFRESID", "CIDADE", "DTCONSULT", "CLINICA", "DTDIAG",
    "BASEDIAG", "TOPOGRUP", "DESCTOPO", "DESCMORFO", "T", "N", "M", "PT",
    "PN", "PM", "S", "G", "PSA", "GLEASON", "LOCALTNM", "IDMITOTIC", "OUTRACLA",
    "META01", "META02", "META03", "META04", "DTTRAT", "NAOTRAT",
    "TRATAMENTO", "TRATHOSP", "TRATFANTES", "TRATFAPOS", "HORMONIO",
    "NENHUMANT", "CIRURANT", "RADIOANT", "QUIMIOANT", "HORMOANT",
    "TMOANT", "IMUNOANT", "OUTROANT", "DTULTINFO", "CICI", "CICIGRUP",
    "CICISUBGRU", "FAIXAETAR", "LATERALI", "INSTORIG", "PERDASEG", "ERRO",
    "DTRECIDIVA", "RECNENHUM", "RECLOCAL", "RECREGIO", "RECDIST", "REC01",
    "REC02", "REC03", "REC04", "CIDO", "DESCIDO", "HABILIT", "HABIT11",
    "HABILIT1", "CIDADEH", "CIDADE_INS", "TMO", "MORFO", "EC",
    "NENHUMAPOS", "CIRURAPOS", "RADIOAPOS", "QUIMIOAPOS", "HORMOAPOS",
    "TMOAPOS", "IMUNOAPOS", "OUTROAPOS", "ULTINFO", "DSCINST", "RACACOR",
    "ESTADOCIVI", "HISTORICOB", "HISTORICOT", "HISTORICOC", "RECESTROG",
    "RECEPROG", "RECHER2", "P16IHQ"
]

df = df.drop(columns=colunas_remover, errors="ignore")
print(f"Colunas restantes: {df.shape[1]}")

Colunas restantes: 26


## Pré-processamento

Após a preparação dos dados, o pré-processamento transforma as variáveis para que fiquem
prontas para treinar um modelo de machine learning. Modelos de ML só trabalham com números,
então todas as colunas precisam ser numéricas, estar na mesma escala e sem valores ausentes.

### 1. Conversão de tipos

As colunas `DRS`, `DRS_INST`, `IBGE`, `IBGEATEN`, `RRAS` e `RRAS_INST` estavam como texto (`object`)
mesmo contendo apenas números. Elas são convertidas para `int64` para que o modelo consiga utilizá-las.
Os 96 valores nulos encontrados em `DRS` são preenchidos com 0 antes da conversão, pois `int` não
aceita valores ausentes.

`INSTITU` é removida pois representa apenas o código identificador do hospital, sem valor preditivo
para o modelo — saber em qual hospital o paciente foi atendido não ajuda a prever óbito.

In [15]:
print(df.dtypes)
print("\n")
print(df.isnull().sum())

INSTITU       object
ESCOLARI       int64
IDADE          int64
SEXO           int64
IBGE          object
CATEATEND      int64
DIAGPREV       int64
TOPO          object
ECGRUP        object
NENHUM         int64
CIRURGIA       int64
RADIO          int64
QUIMIO         int64
IMUNO          int64
OUTROS         int64
CONSDIAG     float64
TRATCONS     float64
DIAGTRAT     float64
ANODIAG        int64
DRS           object
RRAS          object
IBGEATEN       int64
DRS_INST      object
RRAS_INST     object
HABILIT2       int64
obito          int64
dtype: object


INSTITU       0
ESCOLARI      0
IDADE         0
SEXO          0
IBGE          0
CATEATEND     0
DIAGPREV      0
TOPO          0
ECGRUP        0
NENHUM        0
CIRURGIA      0
RADIO         0
QUIMIO        0
IMUNO         0
OUTROS        0
CONSDIAG      0
TRATCONS      0
DIAGTRAT      0
ANODIAG       0
DRS          96
RRAS          0
IBGEATEN      0
DRS_INST      0
RRAS_INST     0
HABILIT2      0
obito         0
dtype: int64


In [16]:
for col in df.select_dtypes(include="object").columns:
    print(f"{col}: {df[col].nunique()} valores únicos — {df[col].unique()[:5]}")

INSTITU: 75 valores únicos — ['000004' '000005' '000008' '000010' '000011']
IBGE: 661 valores únicos — ['3551405' '3543402' '3531308' '3524303' '3553203']
TOPO: 6 valores únicos — ['C349' 'C343' 'C341' 'C348' 'C342']
ECGRUP: 4 valores únicos — ['IV' 'I' 'III' 'II']
DRS: 17 valores únicos — ['13' '05' '17' '01' '16']
RRAS: 20 valores únicos — ['RRAS 13' 'RRAS 17' 'RRAS 01' 'RRAS 06' 'RRAS 05']
DRS_INST: 17 valores únicos — ['13' '17' '01' '06' '07']
RRAS_INST: 18 valores únicos — ['RRAS 13' 'RRAS 17' 'RRAS 06' 'RRAS 09' 'RRAS 05']


In [17]:
# Converter para int — preenchendo nulos com 0 antes
df["DRS"] = df["DRS"].fillna(0).astype(int)
df["DRS_INST"] = df["DRS_INST"].fillna(0).astype(int)
df["IBGE"] = df["IBGE"].fillna(0).astype(int)

# RRAS — extrair número e converter
df["RRAS"] = df["RRAS"].str.extract(r"(\d+)").fillna(0).astype(int)
df["RRAS_INST"] = df["RRAS_INST"].str.extract(r"(\d+)").fillna(0).astype(int)

### 2. Encoding de variáveis categóricas

Variáveis categóricas são aquelas que representam categorias, não quantidades. Existem dois tipos:

- **Ordinal** (`ECGRUP`): tem ordem lógica — estágio I é menos grave que IV. Por isso é mapeada
  diretamente para números: I→1, II→2, III→3, IV→4.

- **Nominal** (`TOPO`): representa subtipos do tumor sem ordem entre si. Por isso usamos `get_dummies`,
  que cria uma coluna binária para cada categoria — o modelo trata cada subtipo de forma independente.

`CONSDIAG`, `DIAGTRAT` e `TRATCONS` são convertidas de `float64` para `int64` pois suas categorias
(0, 1, 2, 3) são números inteiros, sem casas decimais.

In [18]:
# ECGRUP — ordinal
df["ECGRUP"] = df["ECGRUP"].map({"I": 1, "II": 2, "III": 3, "IV": 4})

In [19]:
df = pd.get_dummies(df, columns=["TOPO"], dtype=int)

In [20]:
df = df.drop(columns=["INSTITU"])

In [21]:
df["CONSDIAG"] = df["CONSDIAG"].astype(int)
df["DIAGTRAT"] = df["DIAGTRAT"].astype(int)
df["TRATCONS"] = df["TRATCONS"].astype(int)

In [22]:
print(df.dtypes)
print(f"\nShape final: {df.shape}")

ESCOLARI     int64
IDADE        int64
SEXO         int64
IBGE         int64
CATEATEND    int64
DIAGPREV     int64
ECGRUP       int64
NENHUM       int64
CIRURGIA     int64
RADIO        int64
QUIMIO       int64
IMUNO        int64
OUTROS       int64
CONSDIAG     int64
TRATCONS     int64
DIAGTRAT     int64
ANODIAG      int64
DRS          int64
RRAS         int64
IBGEATEN     int64
DRS_INST     int64
RRAS_INST    int64
HABILIT2     int64
obito        int64
TOPO_C340    int64
TOPO_C341    int64
TOPO_C342    int64
TOPO_C343    int64
TOPO_C348    int64
TOPO_C349    int64
dtype: object

Shape final: (38985, 30)


### 3. Escalonamento

Colunas numéricas podem ter escalas muito diferentes — `IBGE` tem valores na casa dos milhões
enquanto `SEXO` tem apenas 1 ou 2. Sem escalonamento, o modelo daria muito mais peso às colunas
com valores maiores, distorcendo o resultado.

O `StandardScaler` resolve isso transformando cada coluna para ter **média 0 e desvio padrão 1**.
Isso garante que todas as variáveis contribuem de forma equilibrada para o modelo.

A variável alvo `obito` é separada antes do escalonamento pois ela não é uma feature — é o que
o modelo vai aprender a prever.

In [23]:
# Separar features da variável alvo
X = df.drop(columns=["obito"])
y = df["obito"]

# Escalar
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# Voltar para DataFrame
X_scaled = pd.DataFrame(X_scaled, columns=X.columns)

print(X_scaled.describe().round(2))

       ESCOLARI     IDADE      SEXO      IBGE  CATEATEND  DIAGPREV    ECGRUP  \
count  38985.00  38985.00  38985.00  38985.00   38985.00  38985.00  38985.00   
mean       0.00     -0.00     -0.00     -0.00      -0.00     -0.00      0.00   
std        1.00      1.00      1.00      1.00       1.00      1.00      1.00   
min       -1.18     -4.05     -0.79    -47.62      -1.11     -0.73     -2.33   
25%       -0.84     -0.61     -0.79     -0.35      -0.82     -0.73     -0.29   
50%       -0.50      0.04     -0.79      0.21      -0.82     -0.73      0.73   
75%        1.53      0.69      1.27      0.36       1.17      1.36      0.73   
max        1.53      3.29      1.27     41.32       1.17      1.36      0.73   

         NENHUM  CIRURGIA     RADIO  ...  IBGEATEN  DRS_INST  RRAS_INST  \
count  38985.00  38985.00  38985.00  ...  38985.00  38985.00   38985.00   
mean       0.00     -0.00     -0.00  ...      0.00     -0.00       0.00   
std        1.00      1.00      1.00  ...      1.00    